In [2]:
from ecostyles import EcoStyles
# Create styles instance
styles = EcoStyles()

import altair as alt
import pandas as pd

In [3]:
# Register and enable a theme
styles.register_and_enable_theme(theme_name="article")

In [4]:
# Load GDHI data
gdhi_df = pd.read_csv('gross-disposable-household-income-per-head-table-data (1).csv', skiprows=7)
gdhi_df = gdhi_df.drop(columns=['1997', '1998','1999'])
year_cols = gdhi_df.columns[2:]
gdhi_df.columns = list(gdhi_df.columns[:2]) + pd.to_numeric(year_cols, errors='coerce').tolist()
gdhi_df[gdhi_df.columns[2:]] = gdhi_df[gdhi_df.columns[2:]].apply(pd.to_numeric, errors='coerce')
gdhi_df

,Area code,Area name,2000,2001,2002,2003,2004,2005,2006,2007,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,E92000001,England,12188,12653,12920,13296,13827,14322,14909,15619,...,18426,19449,19684,20139,20914,21519,21457,22224,23432,25425
1,N92000002,Northern Ireland,9380,9642,10166,10646,11243,11816,12404,12658,...,14757,15245,15476,15861,16215,16795,16971,17807,18903,20403
2,S92000003,Scotland,10740,11278,11668,12066,12648,13078,13692,14399,...,17189,17715,17856,18185,18619,19361,19345,20000,21133,22908
3,W92000004,Wales,10281,10707,11049,11394,11845,12141,12526,12885,...,15086,15585,15711,16268,16893,17356,17598,18150,18858,20140


In [5]:
# Load GVA data
gva_df = pd.read_csv('gross-value-added-per-hour-worked-table-data (1).csv', skiprows=7)
gva_df

,Area code,Area name,2004,2005,2006,2007,2008,2009,2010,2011,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,E92000001,England,26.07,25.94,27.42,28.41,29.25,29.83,30.32,30.74,...,32.43,33.18,33.95,34.82,35.91,37.11,38.54,40.07,41.50,42.39
1,N92000002,Northern Ireland,20.67,20.57,21.70,22.43,22.98,23.40,23.89,24.48,...,26.23,27.02,27.85,28.52,29.31,30.52,32.26,34.33,35.95,36.88
2,S92000003,Scotland,23.84,23.74,24.91,25.87,26.93,27.90,28.74,29.42,...,31.55,32.18,32.78,33.59,34.71,35.91,37.27,38.75,40.28,41.28
3,W92000004,Wales,21.29,21.23,22.16,22.87,23.53,24.06,24.65,25.27,...,26.73,27.21,27.82,28.66,29.70,30.82,31.93,33.13,34.32,35.15


In [6]:
# Convert GDHI and GVA into long format
gdhi_long_df = gdhi_df.melt(id_vars=['Area code', 'Area name'], var_name='Year', value_name='GDHI')
gva_long_df = gva_df.melt(id_vars=['Area code', 'Area name'], var_name='Year', value_name='GVA')

# Convert year to numeric
gdhi_long_df['Year'] = pd.to_numeric(gdhi_long_df['Year'])
gva_long_df['Year'] = pd.to_numeric(gva_long_df['Year'])

# Merge on area code and year (inner join keeps only overlapping years)
gdhi_gva_df = pd.merge(gdhi_long_df, gva_long_df, on=['Area code', 'Area name', 'Year'], how='inner')
gdhi_gva_df

,Area code,Area name,Year,GDHI,GVA
0,E92000001,England,2004,13827,26.07
1,N92000002,Northern Ireland,2004,11243,20.67
2,S92000003,Scotland,2004,12648,23.84
3,W92000004,Wales,2004,11845,21.29
4,E92000001,England,2005,14322,25.94
...,...,...,...,...,...
75,W92000004,Wales,2022,18858,34.32
76,E92000001,England,2023,25425,42.39
77,N92000002,Northern Ireland,2023,20403,36.88
78,S92000003,Scotland,2023,22908,41.28


In [18]:
title_layer = alt.Chart({'values': [{'x': 0, 'y': 0}]}).mark_text(
    text='Higher Productivity Broadly Means Higher Incomes Across the UK, But Not Equally So',
    align='left',
    baseline='top',
    fontSize=14,
    dx=0
).encode(
    x=alt.value(0),
    y=alt.value(-35)
)

gdhi_gva_scatter = alt.Chart(gdhi_gva_df).mark_circle().encode(
    x=alt.X('GVA:Q', title='Gross Value Added (per hour worked)', scale=alt.Scale(domain=[19,45]), axis=alt.Axis(labelExpr="'£' + datum.value", grid=False)),
    y=alt.Y('GDHI:Q', title='Gross Disposable Household Income (average income per person)', scale=alt.Scale(domain=[10000,26000]), axis=alt.Axis(labelExpr="'£' + format(datum.value, ',.0f')", grid=False)),
    color=alt.Color('Area name:N')
)

gdhi_gva_scatter = alt.layer(title_layer, gdhi_gva_scatter)
styles.add_source(gdhi_gva_scatter, 'ONS')
gdhi_gva_scatter

alt.LayerChart(...)

In [19]:
# Save charts
styles.save(gdhi_gva_scatter, 'visualisation', 'gdhi_gva_scatter', width=450, height=360)